In [1]:
import numpy as np
import pandas as pd
import os
import glob
import itertools
import scanpy as sc
import natsort
import json

from scroutines import basicu

In [2]:
%%time
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/allen_dev_v1/DevVIS_multiome_snRNA_processed.h5ad'
adata_all = sc.read(f, backed='r')
genes_gao = adata_all.var.index.values.astype(str) 
genes_gao, genes_gao.shape

CPU times: user 449 ms, sys: 115 ms, total: 564 ms
Wall time: 1.25 s


(array(['Xkr4', 'Gm1992', 'Gm19938', ..., 'AC133095.1', 'AC234645.1',
        'AC149090.1'], dtype='<U32'),
 (32285,))

In [3]:
outdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_dev_merged'

In [4]:
adata_all.obs['donor_name']

AAACAGCCAATTAAGG-L8XR_210909_02_G08-1131257171    C57BL6J-598371
AAACAGCCAGCAACAG-L8XR_210909_02_G08-1131257171    C57BL6J-598371
AAACATGCAAGGGTTG-L8XR_210909_02_G08-1131257171    C57BL6J-598371
AAACATGCATACTCCT-L8XR_210909_02_G08-1131257171    C57BL6J-598371
AAACCAACACCAGCAT-L8XR_210909_02_G08-1131257171    C57BL6J-598371
                                                       ...      
TTTGTGTTCTCAATGA-L8XR_220721_02_D06-1196254077    C57BL6J-623067
TTTGTGTTCTTAATGG-L8XR_220721_02_D06-1196254077    C57BL6J-623067
TTTGTTGGTATACTGG-L8XR_220721_02_D06-1196254077    C57BL6J-623067
TTTGTTGGTGGGAACA-L8XR_220721_02_D06-1196254077    C57BL6J-623067
TTTGTTGGTTTCGCGC-L8XR_220721_02_D06-1196254077    C57BL6J-623067
Name: donor_name, Length: 200061, dtype: category
Categories (31, object): ['C57BL6J-598371', 'C57BL6J-598855', 'C57BL6J-598856', 'C57BL6J-599582', ..., 'C57BL6J-694053', 'C57BL6J-695497', 'C57BL6J-695500', 'C67BL6J-612863']

In [5]:
a = adata_all.obs.groupby(['age_label', 'donor_name']).size() #.unstack() # .sort_values()
a[a!=0].sort_index(level=0)

age_label  donor_name    
E15.5      C57BL6J-642103     1307
           C57BL6J-642104     1831
E16        C57BL6J-642101     2727
           C57BL6J-642108     3194
E17        C57BL6J-629406     3347
           C57BL6J-629407     3946
           C57BL6J-643216     2161
           C57BL6J-643303     3170
E18        C57BL6J-643307     2795
           C57BL6J-643309     2852
P0         C57BL6J-598855    11556
           C57BL6J-669859     7689
           C57BL6J-669874     5638
           C57BL6J-679646     6810
           C67BL6J-612863     8924
P2         C57BL6J-598856    10139
P4         C57BL6J-689640      384
           C57BL6J-691949     4861
           C57BL6J-694053     6367
P5         C57BL6J-599582    13105
P8         C57BL6J-601610     6298
P9         C57BL6J-638769     5267
P11        C57BL6J-598371     5493
P14        C57BL6J-601611     9693
           C57BL6J-695497    11953
           C57BL6J-695500    16934
P56        C57BL6J-621254     4949
           C57BL6J-622167    

In [6]:
counts = adata_all.obs.groupby(['subclass_label', 'age_label']).size().unstack()
counts

age_label,E15.5,E16,E17,E18,P0,P2,P4,P5,P8,P9,P11,P14,P56,P58
subclass_label,,,,,,,,,,,,,,
ABC NN,0,0,5,1,77,18,51,83,88,33,68,402,384,1
Astro-TE NN,7,15,147,116,1006,450,1024,1224,728,304,748,4607,3187,203
BAM NN,5,3,21,13,33,27,32,42,32,5,16,159,24,0
CGE GABA,105,196,511,199,841,198,103,39,4,0,1,9,0,0
CLA-EPd-CTX Car3 Glut,2,6,48,37,246,32,96,43,10,7,16,248,225,73
CR Glut,6,3,30,16,1,0,1,0,0,0,0,0,0,0
Endo NN,99,96,186,87,215,38,153,91,104,13,70,1230,657,38
Glioblast,86,170,644,357,2458,387,340,349,77,8,89,340,110,5
IMN IT,143,383,2430,1393,11973,1895,526,155,4,0,3,23,1,0


# load full thing

In [7]:
adata_all = sc.read(f)
adata_all

AnnData object with n_obs × n_vars = 200061 × 32285
    obs: 'Unnamed: 0', 'subcluster_id', 'subcluster_label', 'cluster_label', 'subclass_label', 'class_label', 'subcluster_prob', 'cluster_probability', 'subclass_probability', 'class_probability', 'library_prep', 'roi', 'sex', 'donor_name', 'age_label', 'cellNames'

In [8]:
adata_astro = adata_all[adata_all.obs['subclass_label']=='Astro-TE NN']
adata_astro

View of AnnData object with n_obs × n_vars = 13766 × 32285
    obs: 'Unnamed: 0', 'subcluster_id', 'subcluster_label', 'cluster_label', 'subclass_label', 'class_label', 'subcluster_prob', 'cluster_probability', 'subclass_probability', 'class_probability', 'library_prep', 'roi', 'sex', 'donor_name', 'age_label', 'cellNames'

In [9]:
fout = os.path.join(outdir, 'gao25_astro.h5ad')
adata_astro.write(fout)